# Phần dư OOF: độ lớn sai số và cấu trúc tương quan không gian–thời gian

## Cách đọc kết quả

Dùng CSV quan trắc riêng theo hướng dẫn bên dưới. Đọc đồng thời số cặp, độ trễ thời gian và khoảng cách giữa các vị trí trong từng ô variogram trước khi diễn giải sự khác biệt giữa các nhóm độ mặn.

## Mục tiêu
Tách hai câu hỏi trong feedback của thầy:
1. **Độ lớn:** bias, SD, MAE/RMSE của phần dư có khác giữa các nhóm độ mặn không?
2. **Cấu trúc:** ở cùng khoảng cách không gian $h$ **và** độ trễ thời gian $u$, phần dư
   trong nhóm mặn cao có mức tương đồng khác nhóm mặn thấp không?

**Câu hỏi 2 là trọng tâm.** Mục 10–15 tạo variogram thực nghiệm hai chiều cho các cặp
**cùng nhóm**, bao gồm cả cặp khác vị trí **và** khác ngày. Các lát cắt cùng ngày/cùng vị trí
là trường hợp riêng, không thay thế phân tích hai chiều. Notebook chưa có kết luận thực nghiệm
cho đến khi chạy với dữ liệu và cấu hình đã chọn.

## Chạy trên Kaggle
1. Add Input: `Check_BMA_Chuanhoa_LS8_20192020_159.csv`; nếu có nhiều bản, đặt `DATA_PATH`.
2. Chọn CPU; bật Internet nếu thiếu `pyswarm`, XGBoost hoặc LightGBM.
3. `RUN_PROFILE='full'` rồi Run All. `smoke` chỉ kiểm tra code, không dùng kết luận nghiên cứu.
4. Đọc các heatmap/lát cắt ở mục 14 trước, sau đó `readout.md` và bảng CSV trong ZIP Output.
5. Dùng `RESUME_FROM` trỏ tới `checkpoints` cũ để tránh chạy lại PSO khi chỉ đổi chẩn đoán.

## Phương pháp và giả định
- Giữ nested forward OOF: train từ `2019-01-01` đến trước `2020-03-01`; outer train luôn
  đứng trước validation. PSO và chọn blend cây–Ridge chỉ dùng lịch sử của outer fold.
- $e=y-\hat y$; cây được chặn dự báo âm về 0. Phần dư chính `tree`, bổ sung `base` (cây–Ridge).
- Mặc định `GROUP_BY='oof_tree'`: chia nhóm bằng dự báo OOF của **cùng mô hình**, không bằng
  chính phần dư. Có thể chọn `y_true` để mô tả nhóm thực đo; cách này điều kiện hóa trên biến
  đích có thể gây cấu trúc phần dư do chọn nhóm và không phải quy tắc gating dùng lúc dự báo.
- Phân vị 1/3, 2/3 mặc định riêng từng mô hình; không ép tách giá trị bằng nhau. Muốn các
  mô hình cùng ngưỡng, đặt `GROUP_THRESHOLDS=(q1,q2)` theo đơn vị đã xác nhận của dữ liệu.
  Ngưỡng chỉ phục vụ khảo sát, không được dùng để tuyên bố hiệu năng ngoài mẫu.
- Gộp các dòng cùng tọa độ chính xác–ngày thành một đơn vị; tọa độ chưa phải mã trạm đã xác minh.
  Nhóm được gán lại sau gộp bằng cùng ngưỡng. Không có cặp tự ghép, không đếm cả $(i,j)$ và $(j,i)$.
- Khoảng cách haversine (km), độ trễ là chênh lệch **ngày lịch**; không cộng hai đại lượng
  khác đơn vị thành một khoảng cách. Biên bin chung cho mọi nhóm/mô hình, chọn từ hình học,
  không chọn theo hình dạng phần dư. Bin $h=0$ và $u=0$ được tách riêng.
- Chuẩn hóa bằng SD phần dư **của nhóm trên bảng vị trí–ngày**. So sánh cả hình dạng variogram
  chuẩn hóa và tương quan cặp; không suy ra tương quan khác chỉ từ variogram thô cao hơn.
- Mỗi cặp không phải một quan trắc độc lập. Phân tích hai chiều là **mô tả/thăm dò**, không có
  p-value hoặc CI dựa trên bootstrap từng cặp. Bỏ từng outer fold (tính lại SD, giữ ngưỡng/bin)
  và chỉ dùng cặp cùng outer fold là hai kiểm tra độ nhạy, không phải khoảng tin cậy.
- Khác biệt còn có thể do phân bố vị trí/ngày, mùa, bias thay đổi, bin rộng và đổi mô hình giữa
  các outer fold. Kiểm tra độ phủ, khoảng cách trung bình thực trong từng ô và độ nhạy biên bin.
- Các biến PCA/chỉ số đã có trong CSV; nếu từng được học trên toàn dữ liệu, cần tạo lại trong fold
  trước khi diễn giải khả năng tổng quát hóa.

Tham khảo phương pháp: [variogram không gian–thời gian (gstat)](https://r-spatial.github.io/gstat/reference/variogramST.html),
[nested CV (scikit-learn)](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).


## 1. Cấu hình

In [ ]:
from pathlib import Path

DATA_PATH = None  # Ví dụ: '/kaggle/input/check-bma/Check_BMA_Chuanhoa_LS8_20192020_159.csv'
DATA_FILENAME = 'Check_BMA_Chuanhoa_LS8_20192020_159.csv'
RUN_PROFILE = 'full'  # 'full' hoặc 'smoke'
RESUME_FROM = None   # Thư mục checkpoints từ Output cũ, không phải PKL cũ.
DIAGNOSTICS_ONLY = False  # True: chỉ nạp checkpoint đúng cấu hình; thiếu checkpoint sẽ báo lỗi.
AUTO_INSTALL_MISSING = True
MODEL_NAMES = ['Decision Tree', 'Bagging', 'Random Forest', 'PSO Bagging',
               'PSO RF', 'PSO XGB', 'PSO LightGBM']
TRAIN_START, TRAIN_END = '2019-01-01', '2020-03-01'  # TRAIN_END exclusive
USE_SEASONAL = True
RECENCY_HALF_LIFE = None  # Giữ cách tính trọng số thời gian của notebook nguồn.
OOF_FOLDS, OOF_MIN_SCENES = 5, 10
INNER_FOLDS, INNER_MIN_SCENES = 3, 4
PSO_SWARMSIZE, PSO_MAXITER = 12, 10
MODEL_N_JOBS = 2
SEED = 42
PSO_SEEDS = {'PSO RF': 42, 'PSO Bagging': 43, 'PSO XGB': 44,
             'PSO LightGBM': 45, 'Ridge': 46}

# Chỉ thay các tham số dưới đây thì không cần chạy lại PSO; checkpoint vẫn được dùng lại.
N_BOOTSTRAP = 499
TIME_BLOCK_DAYS = 32
SPACE_BLOCK_KM = 40.0
AUTO_LAG_BINS = 4
TIME_BIN_EDGES = None   # Hoặc [0, 16, 32, 64, 128, 365], đơn vị ngày
SPACE_BIN_EDGES = None  # Hoặc [0, 5, 15, 30, 60, 120, 250], đơn vị km
# Ngưỡng đếm tham khảo cho cấu trúc; KHÔNG lọc hình/chênh lệch variogram. Mục 9 có tiêu chí bootstrap riêng.
MIN_GROUP_N = 12
MIN_PAIRS = 20
MIN_DISTINCT_OBS = 12
MIN_CLUSTERS = 4
MIN_BOOT_VALID_FRACTION = 0.8
# Ngưỡng thăm dò để đánh dấu độ lớn khác biệt, không phải ngưỡng chuyên ngành đã xác nhận.
BIAS_EFFECT_THRESHOLD = 0.25  # Đơn vị độ mặn theo CSV
SD_RATIO_THRESHOLD = 1.5
CORR_EFFECT_THRESHOLD = 0.20
FOCUS_MODEL = 'PSO LightGBM'
FOCUS_RESIDUAL = 'tree'  # 'tree' hoặc 'base'
SAVE_ALL_MODEL_FIGURES = True

if RUN_PROFILE not in {'full', 'smoke'}:
    raise ValueError('RUN_PROFILE phải là full hoặc smoke.')
if RUN_PROFILE == 'smoke':
    PSO_SWARMSIZE, PSO_MAXITER, MODEL_N_JOBS = 2, 1, 1
    N_BOOTSTRAP = 39
if len(MODEL_NAMES) != len(set(MODEL_NAMES)) or not MODEL_NAMES:
    raise ValueError('MODEL_NAMES phải có tên mô hình và không lặp.')
if FOCUS_MODEL not in MODEL_NAMES:
    FOCUS_MODEL = MODEL_NAMES[0]
if FOCUS_RESIDUAL not in {'tree', 'base'}:
    raise ValueError('FOCUS_RESIDUAL phải là tree hoặc base.')
if N_BOOTSTRAP < 20 or TIME_BLOCK_DAYS <= 0 or SPACE_BLOCK_KM <= 0:
    raise ValueError('Kiểm tra cấu hình bootstrap/khối.')
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Chẩn đoán không gian–thời gian: thay các tham số này không đổi chữ ký huấn luyện.
DIAGNOSTIC_VERSION = 'joint-st-all-finite-v3'
GROUP_BY = 'oof_tree'  # 'oof_tree' (mặc định) hoặc 'y_true' (thăm dò nhóm thực đo)
GROUP_THRESHOLDS = None  # Hoặc (q1, q2) cố định, dùng chung các mô hình
MIN_PAIR_STATIONS = 3
MIN_PAIR_DATES = 3
MIN_LOO_REPLICATES = 3
NORMALIZED_GAMMA_EFFECT = 0.20  # Chỉ ngưỡng mô tả để ưu tiên xem, không phải kiểm định
MAX_GEOMETRY_PAIRS = 2_000_000  # Chặn O(n^2); không tự lấy mẫu cặp
if GROUP_BY not in {'oof_tree', 'y_true'}:
    raise ValueError('GROUP_BY phải là oof_tree hoặc y_true.')
if GROUP_THRESHOLDS is not None:
    if len(GROUP_THRESHOLDS) != 2 or not all(map(lambda v: float('-inf') < v < float('inf'), GROUP_THRESHOLDS)):
        raise ValueError('GROUP_THRESHOLDS phải chứa 2 giá trị hữu hạn.')
    if GROUP_THRESHOLDS[0] >= GROUP_THRESHOLDS[1]:
        raise ValueError('Cần q1 < q2.')
if AUTO_LAG_BINS < 1 or MIN_PAIR_STATIONS < 1 or MIN_PAIR_DATES < 1:
    raise ValueError('Kiểm tra số bin và độ phủ cặp tối thiểu.')


## 2. Thư viện và tiện ích lưu kết quả

In [ ]:
import hashlib
import importlib
import importlib.metadata
import importlib.util
import inspect
import itertools
import json
import os
import re
import shutil
import subprocess
import sys
import warnings

requirements = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
                'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib', 'pyswarm': 'pyswarm==1.1.2'}
if 'PSO XGB' in MODEL_NAMES:
    requirements['xgboost'] = 'xgboost'
if 'PSO LightGBM' in MODEL_NAMES:
    requirements['lightgbm'] = 'lightgbm'
for module, package in requirements.items():
    if importlib.util.find_spec(module) is None:
        if not AUTO_INSTALL_MISSING:
            raise ImportError(f'Thiếu {module}. Cài {package} hoặc bật AUTO_INSTALL_MISSING và Internet.')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scipy.optimize import minimize_scalar
from sklearn.base import clone
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler
from pyswarm import pso
if 'PSO XGB' in MODEL_NAMES:
    from xgboost import XGBRegressor
if 'PSO LightGBM' in MODEL_NAMES:
    from lightgbm import LGBMRegressor

pd.set_option('display.max_columns', 18)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 160, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
GROUPS = ['low', 'medium', 'high']
GROUP_LABELS = {'low': 'Thấp', 'medium': 'Trung bình', 'high': 'Cao'}
COLORS = {'low': '#0072B2', 'medium': '#E69F00', 'high': '#009E73'}
KINDS = {'tree': 'residual_tree', 'base': 'residual_base'}
PAIR_COMPARISONS = list(itertools.combinations(GROUPS, 2))
PIPELINE_VERSION = 'nested-oof-salinity-diagnostics-v1'

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, (pd.Timestamp, Path)):
        return str(value)
    return value

def save_json(path, value):
    path = Path(path)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(json_safe(value), ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def save_csv(frame, name):
    frame.to_csv(TABLE_DIR / name, index=False, encoding='utf-8-sig')

versions = {name: importlib.metadata.version(name) for name in
            ['numpy', 'pandas', 'scipy', 'scikit-learn', 'matplotlib', 'pyswarm']}
for name in ['xgboost', 'lightgbm']:
    if importlib.util.find_spec(name):
        versions[name] = importlib.metadata.version(name)
print('Chế độ:', RUN_PROFILE, '| PSO:', PSO_SWARMSIZE, 'hạt × tối đa', PSO_MAXITER, 'vòng')
display(pd.Series(versions, name='version').to_frame())

## 3. Đọc dữ liệu train và kiểm tra đơn vị quan trắc

In [ ]:
def resolve_data_path():
    if DATA_PATH is not None:
        path = Path(DATA_PATH)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    if Path('/kaggle/input').is_dir():
        candidates = sorted(Path('/kaggle/input').rglob(DATA_FILENAME))
    else:
        candidates = [Path(DATA_FILENAME)] if Path(DATA_FILENAME).is_file() else []
    if len(candidates) != 1:
        raise FileNotFoundError(f'Tìm thấy {len(candidates)} file {DATA_FILENAME}; đặt DATA_PATH chính xác.')
    return candidates[0]

csv_path = resolve_data_path()
raw = pd.read_csv(csv_path)
raw['sample_id'] = np.arange(len(raw), dtype=int)
date_columns = [c for c in raw if re.fullmatch(r'\d{2}Th\d{4}', c)]
if not date_columns:
    raise ValueError('Thiếu cột ngày dạng 19Th0119.')
if not raw[date_columns].isin([0, 1]).all().all():
    raise ValueError('Cột ngày phải là nhị phân 0/1 và không khuyết.')
active = raw[date_columns].eq(1)
clean = raw.loc[active.sum(axis=1).eq(1)].copy()
clean['date'] = pd.to_datetime(active.loc[clean.index].idxmax(axis=1), format='%dTh%m%y')
train = clean.loc[(clean['date'] >= TRAIN_START) & (clean['date'] < TRAIN_END)].copy()
train = train.sort_values(['date', 'sample_id']).reset_index(drop=True)
train['doy'] = train['date'].dt.dayofyear
train['sin_doy'] = np.sin(2 * np.pi * train['doy'] / 365.25)
train['cos_doy'] = np.cos(2 * np.pi * train['doy'] / 365.25)
train['t'] = (train['date'] - pd.Timestamp(TRAIN_START)).dt.days
TARGET = 'SALINITY_OK'
FEATURES = ['DEM', 'Lat', 'Long', 'EVI', 'NDBI', 'NDMI', 'NDSI', 'NDVI', 'NDWI', 'VSSI', 'SAVI',
            'COSRI', 'EVI2', 'MSI', 'ND23', 'ND47', 'SI1', 'SI2', 'SI3', 'SI4', 'SI5', 'SI6',
            'SI7', 'SI8', 'SI9', 'CA', 'B', 'G', 'R', 'NIR', 'PCA1', 'PCA2', 'PCA3', 'PCA4', 'PCA5']
if USE_SEASONAL:
    FEATURES += ['sin_doy', 'cos_doy', 't']
if train.empty or not np.isfinite(train[FEATURES + [TARGET]].to_numpy(float)).all():
    raise ValueError('Train rỗng hoặc có dữ liệu thiếu/vô cực; kiểm tra CSV trước khi chạy.')
if (train[TARGET] < 0).any() or not train.Lat.between(-90, 90).all() or not train.Long.between(-180, 180).all():
    raise ValueError('Độ mặn/tọa độ ngoài miền hợp lệ.')
locations = train[['Lat', 'Long']].drop_duplicates().sort_values(['Lat', 'Long']).reset_index(drop=True)
locations['station_id'] = [f'S{i:03d}' for i in range(len(locations))]
train = train.merge(locations, on=['Lat', 'Long'], how='left', validate='many_to_one', sort=False)
train = train.sort_values(['date', 'sample_id']).reset_index(drop=True)
X, y, dates = train[FEATURES].to_numpy(float), train[TARGET].to_numpy(float), train.date.to_numpy()
duplicate_keys = train.loc[train.duplicated(['station_id', 'date'], keep=False)].copy()
quality = {'source_path': str(csv_path), 'source_sha256': hashlib.sha256(csv_path.read_bytes()).hexdigest(),
           'raw_rows': len(raw), 'invalid_or_ambiguous_date_rows': int((active.sum(axis=1) != 1).sum()),
           'train_rows': len(train), 'train_dates': train.date.nunique(), 'train_locations': len(locations),
           'extra_station_date_rows': int(train.duplicated(['station_id', 'date']).sum()),
           'target_conflicts_within_station_date': int((train.groupby(['station_id', 'date'])[TARGET].nunique() > 1).sum())}
display(pd.Series(quality, name='value').to_frame())
display(duplicate_keys[['sample_id', 'station_id', 'date', TARGET]].head(12))
print('Các dòng cùng trạm–ngày sẽ được gộp trong phân tích cặp; thống kê theo dòng vẫn được lưu riêng.')

## 4. Tạo fold tiến theo thời gian và định nghĩa mô hình

In [ ]:
def forward_splits(dates, n_folds=5, min_scenes=10):
    """Chia theo ngày quan trắc; train luôn đứng hoàn toàn trước validation."""
    dates = np.asarray(dates, dtype='datetime64[ns]')
    if dates.ndim != 1 or np.isnat(dates).any():
        raise ValueError('dates phải là vector ngày hợp lệ, không có NaT.')
    unique_dates = np.unique(dates)
    if n_folds < 1 or min_scenes < 1 or len(unique_dates) <= min_scenes:
        raise ValueError('Không đủ ngày quan trắc cho forward CV; kiểm tra min_scenes.')
    n_folds = min(n_folds, len(unique_dates) - min_scenes)
    edges = np.linspace(min_scenes, len(unique_dates), n_folds + 1).astype(int)
    result = []
    for a, b in zip(edges[:-1], edges[1:]):
        tr = np.flatnonzero(dates < unique_dates[a])
        va = np.flatnonzero(np.isin(dates, unique_dates[a:b]))
        assert len(tr) and len(va) and dates[tr].max() < dates[va].min()
        result.append((tr, va))
    return result

dt_base = DecisionTreeRegressor(max_depth=7, min_samples_leaf=5, random_state=42)
bagging_base = BaggingRegressor(estimator=clone(dt_base), n_estimators=10, random_state=42, n_jobs=MODEL_N_JOBS)
rf_base = RandomForestRegressor(n_estimators=100, max_depth=7, min_samples_leaf=5, random_state=42, n_jobs=MODEL_N_JOBS)
PSO_SEARCH = {'PSO RF': {'names': ['n_estimators', 'max_depth', 'min_samples_leaf', 'max_features', 'min_samples_split'], 'lower': [120, 3, 3, 0.5, 8], 'upper': [250, 8, 10, 0.8, 20]}, 'PSO Bagging': {'names': ['n_estimators', 'max_depth', 'min_samples_leaf', 'max_features', 'min_samples_split'], 'lower': [120, 3, 3, 0.5, 8], 'upper': [250, 8, 10, 0.8, 20]}, 'PSO XGB': {'names': ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 'colsample_bytree', 'min_child_weight', 'gamma', 'reg_lambda', 'reg_alpha'], 'lower': [50, 2, 0.01, 0.6, 0.5, 3, 0.0, 1.0, 0.0], 'upper': [100, 4, 0.08, 0.85, 0.8, 15, 5.0, 15.0, 5.0]}, 'PSO LightGBM': {'names': ['n_estimators', 'num_leaves', 'learning_rate', 'subsample', 'colsample_bytree', 'min_child_samples', 'reg_lambda', 'reg_alpha'], 'lower': [50, 4, 0.01, 0.6, 0.5, 15, 2.0, 1.0], 'upper': [100, 8, 0.06, 0.8, 0.7, 25, 8.0, 3.0]}, 'Ridge': {'names': ['alpha'], 'lower': [0.1], 'upper': [1000.0]}}
INTEGER_PARAMETERS = {'n_estimators', 'max_depth', 'min_samples_leaf', 'min_samples_split', 'num_leaves', 'min_child_samples'}

def decode_parameters(name, values):
    return {key: int(round(value)) if key in INTEGER_PARAMETERS else float(value) for key, value in zip(PSO_SEARCH[name]['names'], values)}

def build_base_model(name, parameters=None):
    parameters = dict(parameters or {})
    fixed_models = {'Decision Tree': dt_base, 'Bagging': bagging_base, 'Random Forest': rf_base}
    if name in fixed_models:
        model = clone(fixed_models[name])
        if 'n_jobs' in model.get_params():
            model.set_params(n_jobs=MODEL_N_JOBS)
        return model
    if name == 'Ridge':
        return make_pipeline(StandardScaler(), Ridge(random_state=42, **parameters))
    if name == 'PSO RF':
        return RandomForestRegressor(random_state=42, n_jobs=MODEL_N_JOBS, **parameters)
    if name == 'PSO Bagging':
        tree_parameters = {key: parameters.pop(key) for key in ['max_depth', 'min_samples_leaf', 'min_samples_split']}
        tree = DecisionTreeRegressor(random_state=42, **tree_parameters)
        return BaggingRegressor(estimator=tree, bootstrap=True, random_state=42, n_jobs=MODEL_N_JOBS, **parameters)
    if name == 'PSO XGB':
        return XGBRegressor(random_state=42, n_jobs=MODEL_N_JOBS, **parameters)
    if name == 'PSO LightGBM':
        return LGBMRegressor(max_depth=-1, subsample_freq=1, random_state=42, verbose=-1, n_jobs=MODEL_N_JOBS, **parameters)
    raise ValueError('Tên mô hình không hợp lệ: ' + name)

def model_features(model, values):
    """LightGBM receives the same explicit feature names at fit and predict."""
    if 'PSO LightGBM' not in MODEL_NAMES or not isinstance(model, LGBMRegressor):
        return values
    if isinstance(values, pd.DataFrame):
        if list(values.columns) != FEATURES:
            raise ValueError('Tên/thứ tự cột LightGBM phải khớp FEATURES.')
        return values
    values = np.asarray(values)
    if values.ndim != 2 or values.shape[1] != len(FEATURES):
        raise ValueError('Số cột LightGBM không khớp FEATURES.')
    return pd.DataFrame(values, columns=FEATURES)

def predict_history(model, values):
    return model.predict(model_features(model, values))

def fit_history(model, X_fit, y_fit, dates_fit):
    fitted = clone(model)
    X_fit = model_features(fitted, X_fit)
    days = (dates_fit - dates_fit.min()) / np.timedelta64(1, 'D')
    weights = np.ones(len(y_fit)) if RECENCY_HALF_LIFE is None else np.maximum(
        0.5 ** ((days.max() - days) / RECENCY_HALF_LIFE), 0.05)
    if isinstance(fitted, Pipeline):
        fitted.fit(X_fit, y_fit, **{fitted.steps[-1][0] + '__sample_weight': weights})
    else:
        fitted.fit(X_fit, y_fit, sample_weight=weights)
    return fitted

allowed_names = {'Decision Tree', 'Bagging', 'Random Forest'} | set(PSO_SEARCH)
if any(name not in allowed_names or name == 'Ridge' for name in MODEL_NAMES):
    raise ValueError('MODEL_NAMES chứa tên không hợp lệ.')
outer_splits = forward_splits(dates, OOF_FOLDS, OOF_MIN_SCENES)
fold_table = []
for fold, (tr, va) in enumerate(outer_splits, 1):
    # Kiểm tra inner folds trước khi chạy PSO tốn thời gian.
    inner = forward_splits(dates[tr], INNER_FOLDS, INNER_MIN_SCENES)
    fold_table.append({'fold': fold, 'train_start': str(dates[tr].min())[:10],
                       'train_end': str(dates[tr].max())[:10],
                       'oof_start': str(dates[va].min())[:10], 'oof_end': str(dates[va].max())[:10],
                       'n_train': len(tr), 'n_oof': len(va), 'inner_folds': len(inner)})
fold_table = pd.DataFrame(fold_table)
display(fold_table)

## 5. Checkpoint và PSO cục bộ trong mỗi outer fold
Checkpoint chỉ được dùng lại khi dữ liệu, thư viện và cấu hình huấn luyện khớp. Các tham số phân tích không nằm trong chữ ký huấn luyện, nên có thể thay đổi mà không chạy lại cây.

In [ ]:
training_config = {'pipeline': PIPELINE_VERSION, 'profile': RUN_PROFILE, 'models': MODEL_NAMES,
                   'features': FEATURES, 'train_start': TRAIN_START, 'train_end': TRAIN_END,
                   'outer_folds': OOF_FOLDS, 'outer_min_scenes': OOF_MIN_SCENES,
                   'inner_folds': INNER_FOLDS, 'inner_min_scenes': INNER_MIN_SCENES,
                   'swarm': PSO_SWARMSIZE, 'iterations': PSO_MAXITER, 'seeds': PSO_SEEDS,
                   'model_n_jobs': MODEL_N_JOBS, 'recency_half_life': RECENCY_HALF_LIFE,
                   'search': PSO_SEARCH, 'versions': versions}
train_hash = hashlib.sha256(pd.util.hash_pandas_object(
    train[['sample_id', 'date', 'station_id', TARGET] + FEATURES], index=False).values.tobytes()).hexdigest()
signature = hashlib.sha256((json.dumps(training_config, sort_keys=True) + train_hash).encode()).hexdigest()
TRAINING_RUN_DIR = OUTPUT_ROOT / 'oof_residual_diagnostics' / signature[:16]
diagnostic_parameters = {name: globals()[name] for name in [
    'DIAGNOSTIC_VERSION', 'GROUP_BY', 'GROUP_THRESHOLDS', 'N_BOOTSTRAP', 'TIME_BLOCK_DAYS',
    'SPACE_BLOCK_KM', 'AUTO_LAG_BINS', 'TIME_BIN_EDGES', 'SPACE_BIN_EDGES', 'MIN_GROUP_N',
    'MIN_PAIRS', 'MIN_DISTINCT_OBS', 'MIN_PAIR_STATIONS', 'MIN_PAIR_DATES', 'MIN_CLUSTERS',
    'MIN_BOOT_VALID_FRACTION', 'MIN_LOO_REPLICATES', 'NORMALIZED_GAMMA_EFFECT',
    'BIAS_EFFECT_THRESHOLD', 'SD_RATIO_THRESHOLD', 'MAX_GEOMETRY_PAIRS',
    'FOCUS_MODEL', 'FOCUS_RESIDUAL', 'SAVE_ALL_MODEL_FIGURES']}
diagnostic_signature = hashlib.sha256(json.dumps(json_safe(diagnostic_parameters), sort_keys=True).encode()).hexdigest()
RUN_DIR = TRAINING_RUN_DIR / 'diagnostics' / diagnostic_signature[:12]
TABLE_DIR, FIGURE_DIR = RUN_DIR / 'tables', RUN_DIR / 'figures'
CACHE_DIR = TRAINING_RUN_DIR / 'checkpoints'
for folder in [TABLE_DIR, FIGURE_DIR, CACHE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
if RESUME_FROM is not None and not Path(RESUME_FROM).is_dir():
    raise FileNotFoundError('RESUME_FROM phải trỏ tới thư mục checkpoints cũ.')
save_json(RUN_DIR / 'training_config.json', {'signature': signature, 'train_hash': train_hash,
                                          'config': training_config, 'data_quality': quality})
save_csv(fold_table, 'outer_folds.csv')
save_csv(duplicate_keys[['sample_id', 'station_id', 'date', TARGET]], 'repeated_station_date_rows.csv')
tuning_log, fold_log = [], []

def cache_key(fold, name):
    return f'fold_{fold:02d}_{re.sub(r"[^a-z0-9]+", "_", name.lower())}'

def load_checkpoint(key, tr, va):
    for root in [CACHE_DIR] + ([Path(RESUME_FROM)] if RESUME_FROM else []):
        meta_path, array_path = root / f'{key}.json', root / f'{key}.npz'
        if not meta_path.is_file() or not array_path.is_file():
            continue
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        if meta.get('signature') != signature:
            if root != CACHE_DIR:
                print('Bỏ checkpoint khác cấu hình:', key)
            continue
        if meta['train_indices'] != tr.tolist() or meta['validation_indices'] != va.tolist():
            raise ValueError('Checkpoint có chỉ số fold không khớp: ' + key)
        with np.load(array_path, allow_pickle=False) as loaded:
            arrays = {k: loaded[k].copy() for k in loaded.files}
        if any(v.shape != (len(va),) or not np.isfinite(v).all() for v in arrays.values()):
            raise ValueError('Checkpoint hỏng/không hợp lệ: ' + key)
        if root != CACHE_DIR:
            shutil.copy2(meta_path, CACHE_DIR / meta_path.name)
            shutil.copy2(array_path, CACHE_DIR / array_path.name)
        return arrays, meta
    if DIAGNOSTICS_ONLY:
        raise FileNotFoundError('Thiếu checkpoint đúng cấu hình: ' + key)
    return None

def write_checkpoint(key, tr, va, arrays, metadata):
    metadata = dict(metadata, signature=signature, train_indices=tr.tolist(), validation_indices=va.tolist())
    temp = CACHE_DIR / f'{key}.npz.tmp'
    with temp.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    temp.replace(CACHE_DIR / f'{key}.npz')
    save_json(CACHE_DIR / f'{key}.json', metadata)
    return arrays, metadata

def tune_history(name, X_fit, y_fit, dates_fit):
    if name not in PSO_SEARCH:
        return {}, None
    inner = forward_splits(dates_fit, INNER_FOLDS, INNER_MIN_SCENES)
    space = PSO_SEARCH[name]
    def objective(values):
        parameters = decode_parameters(name, values)
        losses = []
        for it, iv in inner:
            fitted = fit_history(build_base_model(name, parameters), X_fit[it], y_fit[it], dates_fit[it])
            losses.append(np.mean((y_fit[iv] - predict_history(fitted, X_fit[iv])) ** 2))
        return float(np.mean(losses))
    kwargs = {'swarmsize': PSO_SWARMSIZE, 'maxiter': PSO_MAXITER}
    if 'seed' in inspect.signature(pso).parameters:
        result = pso(objective, space['lower'], space['upper'], seed=PSO_SEEDS[name], **kwargs)
    else:
        previous_state = np.random.get_state()
        try:
            np.random.seed(PSO_SEEDS[name])
            result = pso(objective, space['lower'], space['upper'], **kwargs)
        finally:
            np.random.set_state(previous_state)
    if hasattr(result, 'x'):
        vector, score = result.x, result.fun
    elif isinstance(result, dict):
        vector, score = result['x'], result['fun']
    else:
        vector, score = result[0], result[1]
    return decode_parameters(name, vector), float(score)

def inner_predictions(name, parameters, X_fit, y_fit, dates_fit):
    prediction = np.full(len(y_fit), np.nan)
    for it, iv in forward_splits(dates_fit, INNER_FOLDS, INNER_MIN_SCENES):
        fitted = fit_history(build_base_model(name, parameters), X_fit[it], y_fit[it], dates_fit[it])
        prediction[iv] = predict_history(fitted, X_fit[iv])
    return prediction

def choose_weight(tree_prediction, ridge_prediction, target):
    valid = np.isfinite(tree_prediction) & np.isfinite(ridge_prediction)
    if valid.sum() < 2:
        raise ValueError('Không đủ inner OOF để chọn trọng số blend.')
    tree_values, ridge_values, truth = tree_prediction[valid], ridge_prediction[valid], target[valid]
    def loss(weight):
        return np.mean((truth - np.clip(weight * tree_values + (1 - weight) * ridge_values, 0, None)) ** 2)
    candidate = minimize_scalar(loss, bounds=(0, 1), method='bounded').x
    return float(min([0.0, float(candidate), 1.0], key=loss))
print('Output:', RUN_DIR)

## 6. Huấn luyện và dự báo nested OOF
Các khối tương lai chỉ được truyền vào bước `predict`. Nhãn của chúng chưa tham gia PSO hoặc chọn trọng số. Tham số, chỉ số train/validation và dự báo được lưu sau từng mô hình/fold.

LightGBM dùng DataFrame có cùng tên/thứ tự cột FEATURES ở cả fit và predict (kể cả các inner fold), để tránh cảnh báo thiếu feature names.

In [ ]:
ridge_oof = np.full(len(train), np.nan)
oof_fold_id = np.full(len(train), -1, dtype=int)
oof_states = {name: {key: np.full(len(train), np.nan) for key in
                    ['tree_raw', 'base', 'w_tree']} for name in MODEL_NAMES}
for fold, (tr, va) in enumerate(outer_splits, 1):
    assert dates[tr].max() < dates[va].min()
    assert (oof_fold_id[va] == -1).all()
    oof_fold_id[va] = fold
    X_history, y_history, date_history = X[tr], y[tr], dates[tr]
    print(f'Outer {fold}/{len(outer_splits)}: train={len(tr)}, dự báo OOF={len(va)}', flush=True)
    ridge_key = cache_key(fold, 'Ridge')
    saved = load_checkpoint(ridge_key, tr, va)
    # Cần inner Ridge để chọn blend nếu còn cây chưa hoàn tất.
    need_inner = any(not (CACHE_DIR / f'{cache_key(fold, name)}.json').is_file() for name in MODEL_NAMES)
    if saved is None:
        ridge_parameters, ridge_score = tune_history('Ridge', X_history, y_history, date_history)
        ridge = fit_history(build_base_model('Ridge', ridge_parameters), X_history, y_history, date_history)
        saved = write_checkpoint(ridge_key, tr, va, {'prediction': predict_history(ridge, X[va])},
                                 {'fold': fold, 'model': 'Ridge', 'parameters': ridge_parameters,
                                  'inner_mse': ridge_score, 'fit_end': str(date_history.max())})
    ridge_arrays, ridge_meta = saved
    ridge_parameters = ridge_meta['parameters']
    ridge_oof[va] = ridge_arrays['prediction']
    tuning_log.append(ridge_meta)
    ridge_inner = None

    for name in MODEL_NAMES:
        key = cache_key(fold, name)
        saved = load_checkpoint(key, tr, va)
        if saved is None:
            if ridge_inner is None:
                ridge_inner = inner_predictions('Ridge', ridge_parameters, X_history, y_history, date_history)
            parameters, score = tune_history(name, X_history, y_history, date_history)
            tree_inner = inner_predictions(name, parameters, X_history, y_history, date_history)
            weight = choose_weight(tree_inner, ridge_inner, y_history)
            fitted = fit_history(build_base_model(name, parameters), X_history, y_history, date_history)
            prediction = predict_history(fitted, X[va])
            arrays = {'tree_raw': prediction,
                      'base': np.clip(weight * prediction + (1 - weight) * ridge_oof[va], 0, None),
                      'w_tree': np.full(len(va), weight)}
            saved = write_checkpoint(key, tr, va, arrays,
                {'fold': fold, 'model': name, 'parameters': parameters, 'inner_mse': score,
                 'ridge_parameters': ridge_parameters, 'w_tree': weight,
                 'fit_end': str(date_history.max()), 'prediction_start': str(dates[va].min()),
                 'prediction_end': str(dates[va].max())})
            print('  Đã fit:', name, '| w_tree =', round(weight, 3), flush=True)
        else:
            print('  Dùng checkpoint:', name, flush=True)
        arrays, meta = saved
        for field in ['tree_raw', 'base', 'w_tree']:
            oof_states[name][field][va] = arrays[field]
        tuning_log.append(meta)
        fold_log.append(meta)

oof_tables = []
for name, state in oof_states.items():
    table = train[['sample_id', 'date', 'station_id', 'Lat', 'Long']].copy()
    table['model'], table['outer_fold'], table['y_true'] = name, oof_fold_id, y
    table['oof_tree_raw'] = state['tree_raw']
    table['oof_tree'] = np.clip(state['tree_raw'], 0, None)
    table['oof_ridge'], table['oof_blend'], table['oof_w_tree'] = ridge_oof, state['base'], state['w_tree']
    table['residual_tree'] = y - table['oof_tree']
    table['residual_base'] = y - table['oof_blend']
    table['valid_oof'] = np.isfinite(table['oof_tree']) & np.isfinite(table['oof_blend'])
    oof_tables.append(table)
oof_rows = pd.concat(oof_tables, ignore_index=True)
save_csv(oof_rows, 'oof_predictions_all_rows.csv')
save_json(RUN_DIR / 'nested_tuning_log.json', tuning_log)
save_json(RUN_DIR / 'outer_fold_log.json', fold_log)
display(oof_rows.groupby('model', sort=False).agg(n_rows=('sample_id', 'size'), n_oof=('valid_oof', 'sum')))

## 7. Chia nhóm độ mặn và gộp vị trí–ngày
Mặc định phân nhóm theo `oof_tree`. Nếu chọn `y_true`, đây là khảo sát có điều kiện trên
độ mặn thực đo, cần thận trọng vì nhãn nhóm chứa chính biến đích tạo phần dư.
Ngưỡng tính trên các dòng OOF hợp lệ, hoặc lấy từ `GROUP_THRESHOLDS`; sau gộp vị trí–ngày
vẫn dùng cùng ngưỡng. Cả hai loại phần dư dùng cùng nhãn nhóm. Không tách tùy ý các dự báo bằng nhau.


In [ ]:
valid_rows = oof_rows.loc[oof_rows.valid_oof].copy()
threshold_records = []

def assign_group(values, q1, q2):
    values = np.asarray(values, float)
    return np.where(values <= q1, 'low', np.where(values <= q2, 'medium', 'high'))

for model in MODEL_NAMES:
    selected = valid_rows.model.eq(model)
    q1, q2 = (np.quantile(valid_rows.loc[selected, GROUP_BY], [1/3, 2/3])
              if GROUP_THRESHOLDS is None else np.asarray(GROUP_THRESHOLDS, float))
    valid_rows.loc[selected, 'salinity_group'] = assign_group(valid_rows.loc[selected, GROUP_BY], q1, q2)
    counts = valid_rows.loc[selected, 'salinity_group'].value_counts()
    threshold_records.append({'model': model, 'group_by': GROUP_BY, 'q1': q1, 'q2': q2,
                              'three_nonempty_groups': q1 < q2 and all(counts.get(g, 0) > 0 for g in GROUPS),
                              **{f'n_{g}': int(counts.get(g, 0)) for g in GROUPS}})
thresholds = pd.DataFrame(threshold_records)
threshold_map = thresholds.set_index('model')[['q1', 'q2']].to_dict('index')

unit_tables = []
for model in MODEL_NAMES:
    rows = valid_rows.loc[valid_rows.model.eq(model)]
    if (rows.groupby(['station_id', 'date']).outer_fold.nunique() > 1).any():
        raise AssertionError('Cùng ngày bị tách sang hai outer fold.')
    unit = rows.groupby(['station_id', 'date'], as_index=False).agg(
        Lat=('Lat', 'first'), Long=('Long', 'first'), outer_fold=('outer_fold', 'first'),
        n_source_rows=('sample_id', 'size'), y_true=('y_true', 'mean'),
        oof_tree=('oof_tree', 'mean'), oof_blend=('oof_blend', 'mean'))
    unit = unit.sort_values(['station_id', 'date']).reset_index(drop=True)
    unit['unit_id'], unit['model'] = np.arange(len(unit)), model
    unit['residual_tree'] = unit.y_true - unit.oof_tree
    unit['residual_base'] = unit.y_true - unit.oof_blend
    cut = threshold_map[model]
    unit['salinity_group'] = assign_group(unit[GROUP_BY], cut['q1'], cut['q2'])
    unit_tables.append(unit)
units = pd.concat(unit_tables, ignore_index=True)
reference = unit_tables[0][['unit_id', 'station_id', 'date', 'Lat', 'Long', 'outer_fold']].copy()
for table in unit_tables[1:]:
    pd.testing.assert_frame_equal(table[reference.columns], reference, check_dtype=False)

# Khối bootstrap: tọa độ km chỉ dùng để xác định ô lưới; khoảng cách cặp dùng haversine phía dưới.
lat_ref, lon_ref = reference.Lat.mean(), reference.Long.mean()
east = (reference.Long - lon_ref) * 111.32 * np.cos(np.radians(lat_ref))
north = (reference.Lat - lat_ref) * 110.57
space_label = list(zip(np.floor(east / SPACE_BLOCK_KM).astype(int), np.floor(north / SPACE_BLOCK_KM).astype(int)))
reference['space_cluster'] = pd.factorize(pd.Series(space_label), sort=True)[0]
reference['time_cluster'] = pd.factorize(((reference.date - reference.date.min()).dt.days // TIME_BLOCK_DAYS), sort=True)[0]
units = units.merge(reference[['unit_id', 'space_cluster', 'time_cluster']], on='unit_id', validate='many_to_one')
cluster_boot = {}
for offset, axis in enumerate(['space_cluster', 'time_cluster']):
    count = reference[axis].nunique()
    rng = np.random.default_rng(SEED + 100 + offset)
    cluster_boot[axis] = rng.multinomial(count, np.full(count, 1/count), size=N_BOOTSTRAP).astype(float)

save_csv(valid_rows, 'oof_rows_grouped.csv')
save_csv(units, 'oof_station_date_grouped.csv')
save_json(RUN_DIR / 'group_thresholds.json', threshold_records)
save_csv(thresholds, 'group_thresholds.csv')
display(thresholds.round(4))
print('Số đơn vị vị trí–ngày mỗi mô hình:', len(reference),
      '| Cụm không gian:', reference.space_cluster.nunique(), '| Khối thời gian:', reference.time_cluster.nunique())
if not thresholds.three_nonempty_groups.all():
    print('LƯU Ý: Có mô hình không tạo được 3 nhóm khác rỗng; không ép tách các giá trị dự báo bằng nhau.')

## 8. Câu hỏi 1 — Độ lớn sai số
Bias, SD, IQR, MAE và RMSE được tính riêng cho từng nhóm/mô hình/loại phần dư, ở cả grain
dòng gốc và vị trí–ngày. **Khác SD chưa chứng minh khác cấu trúc tương quan.**
Bảng theo outer fold giúp kiểm tra nhóm có tập trung vào một giai đoạn dự báo hay không.


In [ ]:
def summarize_groups(frame, grain):
    rows = []
    for model in MODEL_NAMES:
        for kind, residual_column in KINDS.items():
            for group in GROUPS:
                part = frame.loc[frame.model.eq(model) & frame.salinity_group.eq(group)]
                r = part[residual_column].to_numpy(float)
                rows.append({'model': model, 'residual_kind': kind, 'grain': grain, 'group': group,
                    'n': len(part), 'n_stations': part.station_id.nunique(), 'n_dates': part.date.nunique(),
                    'n_outer_folds': part.outer_fold.nunique(),
                    'mean_true': part.y_true.mean(), 'mean_pred_tree': part.oof_tree.mean(),
                    'mean_pred_base': part.oof_blend.mean(), 'bias': np.mean(r) if len(r) else np.nan,
                    'sd': np.std(r, ddof=1) if len(r) > 1 else np.nan,
                    'iqr': np.diff(np.quantile(r, [.25, .75]))[0] if len(r) else np.nan,
                    'mae': np.mean(np.abs(r)) if len(r) else np.nan,
                    'rmse': np.sqrt(np.mean(r*r)) if len(r) else np.nan})
    return pd.DataFrame(rows)

sample_summary = summarize_groups(valid_rows, 'source_row')
unit_summary = summarize_groups(units, 'station_date')
summary_by_fold = []
for fold in sorted(units.outer_fold.unique()):
    part = summarize_groups(units.loc[units.outer_fold.eq(fold)], 'station_date')
    part['outer_fold'] = fold
    summary_by_fold.append(part)
summary_by_fold = pd.concat(summary_by_fold, ignore_index=True)
save_csv(sample_summary, 'group_summary_source_rows.csv')
save_csv(unit_summary, 'group_summary_station_date.csv')
save_csv(summary_by_fold, 'group_summary_by_outer_fold.csv')
display(unit_summary.loc[unit_summary.residual_kind.eq(FOCUS_RESIDUAL),
    ['model', 'group', 'n', 'n_stations', 'n_dates', 'mean_true', 'mean_pred_tree', 'bias', 'sd', 'iqr', 'mae', 'rmse']].round(3))

## 9. Khoảng bất định và so sánh bias/độ phân tán
Lấy mẫu lại toàn bộ cụm không gian hoặc toàn bộ khối ngày, giữ mọi đơn vị trong cụm đi cùng nhau. Hai cách resample là hai kiểm tra độ nhạy riêng, không được diễn giải là một mô hình đầy đủ của mọi phụ thuộc không gian–thời gian. CI của từng nhóm là pointwise. CI cho ba chênh lệch nhóm được hiệu chỉnh đồng thời bằng độ lệch bootstrap lớn nhất trong mỗi mô hình/loại phần dư/chỉ số/cách resample.

Phần này chỉ đánh giá bias/SD. CI ở đây không áp dụng cho variogram hai chiều phía dưới.


In [ ]:
def weighted_moments(weights, values):
    weights = np.atleast_2d(weights)
    total = weights.sum(axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        mean = (weights @ values) / total
        centered_sum = weights @ (values*values) - total * mean*mean
        sd = np.sqrt(np.maximum(centered_sum, 0) / (total - 1))
    mean[total <= 0], sd[total <= 1] = np.nan, np.nan
    return mean, sd

def interval(values):
    values = np.asarray(values)
    valid = values[np.isfinite(values)]
    if len(valid) < max(20, int(N_BOOTSTRAP * MIN_BOOT_VALID_FRACTION)):
        return np.nan, np.nan
    return tuple(np.quantile(valid, [.025, .975]))

def simultaneous_intervals(estimates, draws):
    """Max standardized bootstrap deviation for one explicit family of contrasts."""
    estimates, draws = np.asarray(estimates, float), np.asarray(draws, float)
    lower, upper = np.full(len(estimates), np.nan), np.full(len(estimates), np.nan)
    if not len(estimates):
        return lower, upper
    se = np.array([np.std(column[np.isfinite(column)], ddof=1) if np.isfinite(column).sum() > 1
                   else np.nan for column in draws.T])
    eligible = np.isfinite(estimates) & (se > 1e-12) & (np.isfinite(draws).mean(axis=0) >= MIN_BOOT_VALID_FRACTION)
    if not eligible.any():
        return lower, upper
    # Common replicates retain joint dependence between contrasts.
    matrix = draws[:, eligible]
    common = np.isfinite(matrix).all(axis=1)
    if common.sum() < max(20, int(N_BOOTSTRAP * MIN_BOOT_VALID_FRACTION)):
        return lower, upper
    maxima = np.max(np.abs((matrix[common] - estimates[eligible]) / se[eligible]), axis=1)
    critical = np.quantile(maxima, .95)
    lower[eligible] = estimates[eligible] - critical * se[eligible]
    upper[eligible] = estimates[eligible] + critical * se[eligible]
    return lower, upper

group_intervals, group_contrasts = [], []
for model in MODEL_NAMES:
    model_units = units.loc[units.model.eq(model)].sort_values('unit_id')
    for kind, residual_column in KINDS.items():
        for axis in ['space_cluster', 'time_cluster']:
            estimates, draws, enough = {}, {}, {}
            for group in GROUPS:
                part = model_units.loc[model_units.salinity_group.eq(group)]
                r = part[residual_column].to_numpy(float)
                clusters = part[axis].to_numpy(int)
                enough[group] = len(part) >= MIN_GROUP_N and part[axis].nunique() >= MIN_CLUSTERS
                boot_bias, boot_sd = weighted_moments(cluster_boot[axis][:, clusters], r)
                point = weighted_moments(np.ones((1, len(r))), r)
                estimates[group] = {'bias': point[0][0], 'sd': point[1][0]}
                draws[group] = {'bias': boot_bias, 'sd': boot_sd}
                for metric in ['bias', 'sd']:
                    lo, hi = interval(draws[group][metric]) if enough[group] else (np.nan, np.nan)
                    group_intervals.append({'model': model, 'residual_kind': kind, 'group': group,
                        'metric': metric, 'bootstrap_axis': axis, 'estimate': estimates[group][metric],
                        'ci_low': lo, 'ci_high': hi, 'n_clusters': part[axis].nunique(),
                        'supported': enough[group], 'ci_scope': 'pointwise_conditional_on_oof'})

            for metric in ['bias_difference', 'sd_ratio']:
                records, bootstrap_columns, point_values = [], [], []
                for g1, g2 in PAIR_COMPARISONS:
                    if metric == 'bias_difference':
                        point = estimates[g2]['bias'] - estimates[g1]['bias']
                        boot = draws[g2]['bias'] - draws[g1]['bias']
                    else:
                        with np.errstate(divide='ignore', invalid='ignore'):
                            point = np.log(estimates[g2]['sd'] / estimates[g1]['sd'])
                            boot = np.log(draws[g2]['sd'] / draws[g1]['sd'])
                        boot[~np.isfinite(boot)] = np.nan
                    supported = enough[g1] and enough[g2] and np.isfinite(point)
                    records.append({'model': model, 'residual_kind': kind, 'metric': metric,
                                    'group_1': g1, 'group_2': g2, 'bootstrap_axis': axis,
                                    'supported': supported})
                    point_values.append(point if supported else np.nan)
                    bootstrap_columns.append(boot if supported else np.full(N_BOOTSTRAP, np.nan))
                lo, hi = simultaneous_intervals(point_values, np.column_stack(bootstrap_columns))
                for j, record in enumerate(records):
                    point = point_values[j]
                    if metric == 'sd_ratio':
                        point, lower, upper = np.exp(point), np.exp(lo[j]), np.exp(hi[j])
                        practical = np.isfinite(point) and (point >= SD_RATIO_THRESHOLD or point <= 1/SD_RATIO_THRESHOLD)
                        excludes_null = np.isfinite(lower) and (lower > 1 or upper < 1)
                    else:
                        lower, upper = lo[j], hi[j]
                        practical = np.isfinite(point) and abs(point) >= BIAS_EFFECT_THRESHOLD
                        excludes_null = np.isfinite(lower) and (lower > 0 or upper < 0)
                    record.update(estimate=point, ci_low=lower, ci_high=upper,
                                  practical_effect=practical, excludes_null=excludes_null,
                                  ci_scope='simultaneous_3_group_contrasts_conditional_on_oof')
                    group_contrasts.append(record)
group_intervals = pd.DataFrame(group_intervals)
group_contrasts = pd.DataFrame(group_contrasts)
save_csv(group_intervals, 'group_bootstrap_intervals.csv')
save_csv(group_contrasts, 'group_contrasts.csv')
display(group_contrasts.loc[group_contrasts.model.eq(FOCUS_MODEL) &
    group_contrasts.residual_kind.eq(FOCUS_RESIDUAL)].round(3))

## 10. Hình học cặp theo đồng thời không gian và thời gian
Xét toàn bộ cặp vị trí–ngày duy nhất, sau đó mới giữ cặp cùng nhóm trong mục 11.
Ba loại cặp: $h=0,u>0$ (cùng vị trí), $h>0,u=0$ (cùng ngày), $h>0,u>0$ (khác cả hai).
Bin 0 là **đúng bằng 0**; các bin dương có dạng $(a,b]$. Ô $(0,0)$ không có cặp sau gộp,
không được điền giả một điểm variogram bằng 0. Bảng biên bin gồm cả ô rỗng để không nối qua dữ liệu thiếu.


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    a1, a2 = np.radians(lat1), np.radians(lat2)
    da, dl = a2 - a1, np.radians(lon2 - lon1)
    a = np.sin(da/2)**2 + np.cos(a1) * np.cos(a2) * np.sin(dl/2)**2
    return 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def make_st_bins(values, configured, domain):
    values = np.asarray(values, float)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError('Khoảng cách phải hữu hạn và không âm.')
    positive = values[values > 0]
    if configured is None:
        edges = (np.unique(np.r_[0., np.quantile(positive, np.linspace(0, 1, AUTO_LAG_BINS+1)[1:])])
                 if len(positive) else np.array([0.]))
    else:
        edges = np.asarray(configured, float)
        if edges.ndim != 1 or len(edges) < 2 or not np.isfinite(edges).all() or edges[0] != 0 or not (np.diff(edges) > 0).all():
            raise ValueError('Biên bin phải hữu hạn, tăng nghiêm ngặt và bắt đầu đúng từ 0.')
    ids = np.full(len(values), -1, dtype=int)
    ids[values == 0] = 0
    if len(edges) > 1:
        candidates = np.asarray(pd.cut(values, edges, labels=False, right=True), float)
        good = np.isfinite(candidates) & (values > 0)
        ids[good] = candidates[good].astype(int) + 1
    records = [dict(domain=domain, lag_bin=0, lower=0., upper=0., center=0., label='0',
                    n_geometry_pairs=int((ids == 0).sum()), unit='km' if domain == 'space' else 'days')]
    for k, (a, b) in enumerate(zip(edges[:-1], edges[1:]), 1):
        records.append(dict(domain=domain, lag_bin=k, lower=float(a), upper=float(b), center=float((a+b)/2),
                            label=f'({a:.4g}, {b:.4g}]', n_geometry_pairs=int((ids == k).sum()),
                            unit='km' if domain == 'space' else 'days'))
    return ids, pd.DataFrame(records), edges

reference = reference.sort_values('unit_id').reset_index(drop=True)
assert np.array_equal(reference.unit_id, np.arange(len(reference)))
n_candidates = len(reference) * (len(reference)-1) // 2
if n_candidates > MAX_GEOMETRY_PAIRS:
    raise ValueError(f'{n_candidates:,} cặp vượt MAX_GEOMETRY_PAIRS; tăng giới hạn có chủ đích hoặc thu hẹp phạm vi dữ liệu.')
i, j = np.triu_indices(len(reference), k=1)
geometry = pd.DataFrame({'i': i, 'j': j})
geometry['distance_km'] = haversine_km(reference.Lat.to_numpy()[i], reference.Long.to_numpy()[i],
                                       reference.Lat.to_numpy()[j], reference.Long.to_numpy()[j])
geometry['time_lag_days'] = np.abs((reference.date.to_numpy()[j] - reference.date.to_numpy()[i]) / np.timedelta64(1, 'D'))
geometry['same_outer_fold'] = reference.outer_fold.to_numpy()[i] == reference.outer_fold.to_numpy()[j]
geometry['pair_domain'] = np.select([geometry.distance_km.eq(0), geometry.time_lag_days.eq(0)],
                                    ['time_only', 'space_only'], default='joint')
space_ids, space_bins, space_edges = make_st_bins(geometry.distance_km, SPACE_BIN_EDGES, 'space')
time_ids, time_bins, time_edges = make_st_bins(geometry.time_lag_days, TIME_BIN_EDGES, 'time')
geometry['space_bin'], geometry['time_bin'] = space_ids, time_ids
st_lag_bins = pd.concat([space_bins, time_bins], ignore_index=True)
excluded_geometry = int(((space_ids < 0) | (time_ids < 0)).sum())
geometry = geometry.loc[(space_ids >= 0) & (time_ids >= 0)].reset_index(drop=True)
save_csv(geometry, 'st_pair_geometry.csv')
save_csv(st_lag_bins, 'st_lag_bins.csv')
save_json(RUN_DIR / 'st_geometry_summary.json', {'candidate_pairs': n_candidates,
    'retained_pairs': len(geometry), 'outside_configured_bins': excluded_geometry,
    'space_edges': space_edges, 'time_edges': time_edges})
display(st_lag_bins.round(3))
print('Tất cả cặp:', n_candidates, '| Ngoài biên cấu hình:', excluded_geometry)
display(geometry.groupby('pair_domain').size().rename('n_pairs').to_frame())


## 11. Giữ cặp trong từng nhóm; kiểm tra độ phủ
Cặp thấp–thấp, trung bình–trung bình, cao–cao được phân tích riêng. Cặp khác nhóm được đếm
nhưng không trộn vào variogram trong nhóm. Số cặp nhiều không đồng nghĩa nhiều quan trắc độc lập.
Bảng vị trí–ngày lưu thêm phần dư chuẩn hóa; nhóm SD bằng 0 không có variogram chuẩn hóa.


In [ ]:
for kind, column in KINDS.items():
    strata = units.groupby(['model', 'salinity_group'])[column]
    mu, sd = strata.transform('mean'), strata.transform('std')
    units[f'z_{kind}'] = (units[column] - mu) / sd.where(sd > 1e-12)
save_csv(units, 'oof_station_date_grouped.csv')
st_pair_tables, pair_audit = [], []
for model in MODEL_NAMES:
    point = units.loc[units.model.eq(model)].sort_values('unit_id').reset_index(drop=True)
    pair = geometry.copy()
    i, j = pair.i.to_numpy(int), pair.j.to_numpy(int)
    gi, gj = point.salinity_group.to_numpy()[i], point.salinity_group.to_numpy()[j]
    same = gi == gj
    audit = pair.assign(same_group=same).groupby(['pair_domain', 'same_group']).size().reset_index(name='n_pairs')
    audit['model'] = model
    pair_audit.append(audit)
    pair = pair.loc[same].copy()
    pair['model'], pair['salinity_group'] = model, gi[same]
    for side in ['i', 'j']:
        ids = pair[side].to_numpy(int)
        for column in ['station_id', 'date', 'outer_fold', 'residual_tree', 'residual_base', 'z_tree', 'z_base']:
            pair[f'{column}_{side}'] = point[column].to_numpy()[ids]
    st_pair_tables.append(pair)
st_pairs = pd.concat(st_pair_tables, ignore_index=True)
pair_audit = pd.concat(pair_audit, ignore_index=True)
group_coverage = units.groupby(['model', 'salinity_group']).agg(
    n_units=('unit_id', 'size'), n_stations=('station_id', 'nunique'),
    n_dates=('date', 'nunique'), n_outer_folds=('outer_fold', 'nunique'),
    date_min=('date', 'min'), date_max=('date', 'max')).reset_index()
coverage_by_date = units.groupby(['model', 'date', 'outer_fold', 'salinity_group']).size().reset_index(name='n_units')
save_csv(st_pairs, 'st_within_group_pairs.csv')
save_csv(pair_audit, 'st_pair_membership_audit.csv')
save_csv(group_coverage, 'st_group_coverage.csv')
save_csv(coverage_by_date, 'st_group_coverage_by_date.csv')
display(group_coverage.loc[group_coverage.model.eq(FOCUS_MODEL)])
display(pair_audit.loc[pair_audit.model.eq(FOCUS_MODEL)])
if GROUP_BY == 'y_true':
    print('LƯU Ý: phân nhóm bằng thực đo chỉ là khảo sát mô tả; nhãn nhóm chứa biến đích.')


## 12. Variogram hai chiều: thô, chuẩn hóa và tương quan cặp
Với cặp $P_g(h,u)$ có cả hai đầu thuộc nhóm $g$:

$$\hat\gamma_g(h,u)=\frac{1}{2|P_g(h,u)|}\sum_{(i,j)\in P_g(h,u)}(e_i-e_j)^2,$$
$$z_i=(e_i-\bar e_g)/s_g,\qquad
\hat\gamma_g^{std}(h,u)=\tfrac12\operatorname{mean}(z_i-z_j)^2=\hat\gamma_g(h,u)/s_g^2.$$

`correlation` là Pearson trên hai vector ghép đối xứng $[e_i,e_j]$ và $[e_j,e_i]$,
không phụ thuộc cách đánh số hai đầu cặp. Đây là tương quan thực nghiệm theo ô, không phải
tương quan giữa phần dư với khoảng cách. Với quá trình dừng bậc hai trong nhóm,
$\gamma_g(h,u)/\sigma_g^2=1-\rho_g(h,u)$; **không áp dụng đẳng thức đó như một phép đổi
chính xác giữa hai ước lượng mẫu**, vì các đầu cặp trong từng ô có thể có phân bố khác nhau.

**Hiển thị mọi giá trị hữu hạn**, kể cả ô chỉ có một cặp. Không lọc hình theo 20 cặp/12 quan trắc.
`supported`/`coverage_supported` chỉ là cờ đếm tham khảo trong CSV, không chứng minh độ tin cậy.
Ô xám/KXĐ chỉ dùng khi không có cặp hoặc chỉ số không xác định (ví dụ SD nhóm bằng 0).
`all` dùng mọi cặp; `same_outer_fold` bỏ cặp qua ranh giới mô hình OOF để kiểm tra độ nhạy.


In [ ]:
def symmetric_pair_correlation(left, right):
    left, right = np.asarray(left, float), np.asarray(right, float)
    if len(left) < 2:
        return np.nan
    mean = (left.mean() + right.mean()) / 2
    variance = np.mean((left*left + right*right)/2) - mean*mean
    if variance <= 1e-12:
        return np.nan
    return float(np.clip((np.mean(left*right) - mean*mean) / variance, -1, 1))

def aggregate_st_pairs(pair_frame, point_frame, pair_scope='all'):
    if pair_scope not in {'all', 'same_outer_fold'}:
        raise ValueError('pair_scope không hợp lệ.')
    selected = pair_frame if pair_scope == 'all' else pair_frame.loc[pair_frame.same_outer_fold]
    strata = dict(tuple(selected.groupby(['model', 'salinity_group', 'space_bin', 'time_bin'], sort=False)))
    point_strata = dict(tuple(point_frame.groupby(['model', 'salinity_group'], sort=False)))
    records = []
    for model, kind, group in itertools.product(MODEL_NAMES, KINDS, GROUPS):
        points = point_strata.get((model, group), point_frame.iloc[:0])
        sd = points[KINDS[kind]].std(ddof=1)
        for sb, tb in itertools.product(space_bins.lag_bin, time_bins.lag_bin):
            part = strata.get((model, group, sb, tb), selected.iloc[:0])
            n = len(part)
            obs = len(set(part.i) | set(part.j))
            stations = len(set(part.station_id_i) | set(part.station_id_j))
            dates_n = len(set(part.date_i) | set(part.date_j))
            folds_n = len(set(part.outer_fold_i) | set(part.outer_fold_j))
            left, right = (part[f'{KINDS[kind]}_{side}'].to_numpy(float) for side in ['i', 'j'])
            gamma = float(np.mean((left-right)**2)/2) if n else np.nan
            normalized = gamma / sd**2 if np.isfinite(sd) and sd > 1e-12 else np.nan
            corr = symmetric_pair_correlation(left, right)
            coverage_ok = (len(points) >= MIN_GROUP_N and n >= MIN_PAIRS and obs >= MIN_DISTINCT_OBS
                           and stations >= MIN_PAIR_STATIONS and dates_n >= MIN_PAIR_DATES)
            reasons = []
            for bad, message in [(len(points) < MIN_GROUP_N, 'small_group'), (n < MIN_PAIRS, 'few_pairs'),
                (obs < MIN_DISTINCT_OBS, 'few_distinct_observations'), (stations < MIN_PAIR_STATIONS, 'few_stations'),
                (dates_n < MIN_PAIR_DATES, 'few_dates')]:
                if bad:
                    reasons.append(message)
            records.append(dict(model=model, residual_kind=kind, salinity_group=group, pair_scope=pair_scope,
                space_bin=int(sb), time_bin=int(tb), pair_domain='time_only' if sb == 0 and tb > 0 else
                    'space_only' if tb == 0 and sb > 0 else 'joint' if sb > 0 and tb > 0 else 'origin',
                n_pairs=n, n_observations=obs, n_stations=stations, n_dates=dates_n, n_outer_folds=folds_n,
                n_group_units=len(points), group_residual_sd=sd, semivariance=gamma,
                semivariance_standardized=normalized, correlation=corr, coverage_supported=coverage_ok,
                supported=coverage_ok and np.isfinite(normalized),
                correlation_supported=coverage_ok and np.isfinite(corr),
                estimate_available=np.isfinite(normalized), correlation_available=np.isfinite(corr),
                support_note=';'.join(reasons) + (';zero_or_undefined_group_sd' if not np.isfinite(normalized) and n else ''),
                mean_distance_km=part.distance_km.mean(), mean_time_lag_days=part.time_lag_days.mean(),
                fraction_cross_fold=float((~part.same_outer_fold).mean()) if n else np.nan))
    return pd.DataFrame(records)

st_surfaces = pd.concat([aggregate_st_pairs(st_pairs, units, scope)
                        for scope in ['all', 'same_outer_fold']], ignore_index=True)
save_csv(st_surfaces, 'st_variogram_surfaces.csv')
focus_surface = st_surfaces.loc[st_surfaces.model.eq(FOCUS_MODEL) &
    st_surfaces.residual_kind.eq(FOCUS_RESIDUAL) & st_surfaces.pair_scope.eq('all')]
display(focus_surface.loc[focus_surface.n_pairs.gt(0), ['salinity_group', 'space_bin', 'time_bin',
    'n_pairs', 'n_observations', 'n_stations', 'n_dates', 'semivariance',
    'semivariance_standardized', 'correlation', 'supported', 'fraction_cross_fold']].round(3))


## 13. So sánh giữa nhóm ở cùng ô khoảng cách và kiểm tra độ nhạy
Chênh lệch luôn là **nhóm 2 − nhóm 1**; tính và hiển thị khi cả hai nhóm có giá trị hữu hạn trong cùng ô, không lọc theo ngưỡng đếm.
Giữ khoảng cách trung bình thực của cả hai nhóm để nhận ra việc cùng bin nhưng phân bố khoảng cách khác nhau.

Bỏ từng outer fold: loại cặp nếu một đầu thuộc fold đó; giữ ngưỡng nhóm/biên bin,
**tính lại SD nhóm** trên các đơn vị còn lại. Khoảng min–max là độ nhạy của các lần bỏ fold,
**không phải CI 95%**. Cùng dấu sau bỏ fold cũng chưa phải kiểm định khác biệt.


In [ ]:
ST_METRICS = ['semivariance', 'semivariance_standardized', 'correlation']
def contrast_st_surfaces(surface):
    records = []
    keys = ['model', 'residual_kind', 'pair_scope', 'space_bin', 'time_bin']
    for key, part in surface.groupby(keys, sort=False):
        indexed = part.set_index('salinity_group')
        for g1, g2 in PAIR_COMPARISONS:
            a, b = indexed.loc[g1], indexed.loc[g2]
            for metric in ST_METRICS:
                flag = ('coverage_supported' if metric == 'semivariance' else
                        'correlation_supported' if metric == 'correlation' else 'supported')
                available = bool(np.isfinite(a[metric]) and np.isfinite(b[metric]))
                supported = bool(a[flag] and b[flag] and available)
                record = dict(zip(keys, key), group_1=g1, group_2=g2, metric=metric,
                    pair_domain=a.pair_domain, supported=supported, available=available,
                    estimate=float(b[metric]-a[metric]) if available else np.nan,
                    n_pairs_1=a.n_pairs, n_pairs_2=b.n_pairs,
                    n_observations_1=a.n_observations, n_observations_2=b.n_observations,
                    mean_distance_km_1=a.mean_distance_km, mean_distance_km_2=b.mean_distance_km,
                    mean_time_lag_days_1=a.mean_time_lag_days, mean_time_lag_days_2=b.mean_time_lag_days)
                records.append(record)
    return pd.DataFrame(records)

st_contrasts = contrast_st_surfaces(st_surfaces)
loo_tables, loo_contrast_tables = [], []
for omitted in sorted(units.outer_fold.unique()):
    reduced_points = units.loc[~units.outer_fold.eq(omitted)]
    reduced_pairs = st_pairs.loc[~st_pairs.outer_fold_i.eq(omitted) & ~st_pairs.outer_fold_j.eq(omitted)]
    sub = aggregate_st_pairs(reduced_pairs, reduced_points, 'all')
    sub['omitted_fold'] = int(omitted)
    loo_tables.append(sub)
    contrast = contrast_st_surfaces(sub)
    contrast['omitted_fold'] = int(omitted)
    loo_contrast_tables.append(contrast)
st_loo = pd.concat(loo_tables, ignore_index=True)
st_loo_contrasts = pd.concat(loo_contrast_tables, ignore_index=True)
contrast_keys = ['model', 'residual_kind', 'pair_scope', 'space_bin', 'time_bin', 'group_1', 'group_2', 'metric']
loo_lookup = dict(tuple(st_loo_contrasts.groupby(contrast_keys, sort=False)))
sensitivity_records = []
for _, row in st_contrasts.loc[st_contrasts.pair_scope.eq('all')].iterrows():
    key = tuple(row[k] for k in contrast_keys)
    part = loo_lookup.get(key, st_loo_contrasts.iloc[:0])
    values = part.loc[part.available, 'estimate'].dropna().to_numpy()
    agreement = float(np.mean(np.sign(values) == np.sign(row.estimate))) if len(values) and np.isfinite(row.estimate) and abs(row.estimate) > 1e-12 else np.nan
    stable = len(values) >= MIN_LOO_REPLICATES and np.isfinite(agreement) and agreement >= .8
    sensitivity_records.append(dict(zip(contrast_keys, key), n_loo_available=len(values),
        loo_min=float(values.min()) if len(values) else np.nan,
        loo_max=float(values.max()) if len(values) else np.nan,
        loo_direction_agreement=agreement, loo_stable_direction=stable))
st_sensitivity = pd.DataFrame(sensitivity_records)
st_contrasts = st_contrasts.merge(st_sensitivity, on=contrast_keys, how='left', validate='one_to_one')
save_csv(st_contrasts, 'st_group_contrasts.csv')
save_csv(st_loo, 'st_leave_one_fold_out_surfaces.csv')
save_csv(st_loo_contrasts, 'st_leave_one_fold_out_contrasts.csv')
display(st_contrasts.loc[st_contrasts.model.eq(FOCUS_MODEL) & st_contrasts.residual_kind.eq(FOCUS_RESIDUAL) &
    st_contrasts.pair_scope.eq('all') & st_contrasts.group_1.eq('low') & st_contrasts.group_2.eq('high') &
    st_contrasts.metric.eq('semivariance_standardized'), ['space_bin', 'time_bin', 'estimate',
        'available', 'n_pairs_1', 'n_pairs_2', 'n_loo_available', 'loo_min', 'loo_max', 'loo_direction_agreement']].round(3))


## 14. Biểu đồ chính — So sánh cấu trúc giữa ba nhóm
Heatmap dùng cùng biên bin và thang màu giữa các nhóm. Trục ngang là khoảng cách không gian,
trục dọc là độ trễ thời gian. Mỗi ô ghi giá trị, `n` cặp và `N` quan trắc duy nhất; ô xám/KXĐ là không có cặp hoặc chỉ số không xác định,
**không phải giá trị 0**. Các ô khoảng cách có thể rộng không bằng nhau, nên heatmap hiển thị
theo danh mục khoảng; xem nhãn và bảng bin trước khi nhận xét tốc độ biến đổi.


In [ ]:
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.patches import Rectangle

INDIVIDUAL_FIGURE_DIR = FIGURE_DIR / 'st'
INDIVIDUAL_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
ST_LABELS = {'semivariance': 'Bán phương sai thô (đơn vị độ mặn²)',
             'semivariance_standardized': 'Bán phương sai chuẩn hóa (không đơn vị)',
             'correlation': 'Tương quan phần dư theo ô'}
LINE_STYLES = {'low': '-', 'medium': '--', 'high': ':'}
MARKERS = {'low': 'o', 'medium': 's', 'high': '^'}

def figure_title(model, kind, text):
    return f'{text}\n{model} | phần dư {kind} | nhóm theo {GROUP_BY} | {RUN_PROFILE}'

def save_st_figure(fig, model, kind, name, show=True):
    slug = re.sub(r'[^a-z0-9]+', '_', model.lower())
    short_name = name.replace('semivariance_standardized', 'gamma_std').replace('semivariance', 'gamma_raw').replace('same_outer_fold', 'samefold').replace('normalized_group_differences', 'group_diff')
    fig.savefig(INDIVIDUAL_FIGURE_DIR / f'{slug}_{kind}_{short_name}.png', bbox_inches='tight')
    if show:
        display(fig)
    plt.close(fig)

def metric_support(part, metric):
    """Mathematical availability only; count thresholds never mask plotted values."""
    return np.isfinite(part[metric])

def set_st_axes(ax):
    ax.set_xticks(range(len(space_bins)), space_bins.label, rotation=35, ha='right')
    ax.set_yticks(range(len(time_bins)), time_bins.label)
    ax.set_xlabel('Khoảng cách không gian h (km)')
    ax.set_ylabel('Độ trễ thời gian u (ngày)')

def plot_st_heatmaps(model=FOCUS_MODEL, kind=FOCUS_RESIDUAL,
                     metric='semivariance_standardized', pair_scope='all', show=True):
    part = st_surfaces.loc[st_surfaces.model.eq(model) & st_surfaces.residual_kind.eq(kind) & st_surfaces.pair_scope.eq(pair_scope)]
    good = metric_support(part, metric)
    values = part.loc[good, metric].to_numpy()
    norm = Normalize(-1, 1) if metric == 'correlation' else Normalize(0, max(float(values.max()) if len(values) else 1, 1e-9))
    cmap = plt.get_cmap('RdBu_r' if metric == 'correlation' else 'viridis').copy()
    cmap.set_bad('#e5e7eb')
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.7), layout='constrained', squeeze=False)
    for ax, group in zip(axes.flat, GROUPS):
        sub = part.loc[part.salinity_group.eq(group)].copy()
        sub['shown'] = sub[metric].where(metric_support(sub, metric))
        grid = sub.pivot(index='time_bin', columns='space_bin', values='shown').reindex(index=time_bins.lag_bin, columns=space_bins.lag_bin)
        img = ax.imshow(np.ma.masked_invalid(grid.to_numpy(float)), origin='lower', aspect='auto', cmap=cmap, norm=norm)
        for _, row in sub.iterrows():
            val = row['shown']
            label = (f'{val:.2f}\nn={int(row.n_pairs)}\nN={int(row.n_observations)}' if np.isfinite(val)
                     else f'KXĐ\nn={int(row.n_pairs)}\nN={int(row.n_observations)}')
            color = ('white' if (abs(val) > .65 if metric == 'correlation' else norm(val) < .45) else '#111827') if np.isfinite(val) else '#111827'
            ax.text(row.space_bin, row.time_bin, label, ha='center', va='center', fontsize=8, color=color)
        ax.set_title(f'Nhóm {GROUP_LABELS[group]}')
        set_st_axes(ax)
    fig.colorbar(img, ax=list(axes.flat), shrink=.8, label=ST_LABELS[metric])
    fig.suptitle(figure_title(model, kind, f'{ST_LABELS[metric]} | cặp: {pair_scope}'))
    fig.supxlabel('n: số cặp; N: số quan trắc duy nhất. Xám/KXĐ: không có cặp hoặc giá trị không xác định; không lọc theo ngưỡng đếm.', fontsize=9)
    save_st_figure(fig, model, kind, f'heatmap_{metric}_{pair_scope}', show)

def plot_st_counts(model=FOCUS_MODEL, kind=FOCUS_RESIDUAL, show=True):
    part = st_surfaces.loc[st_surfaces.model.eq(model) & st_surfaces.residual_kind.eq(kind) & st_surfaces.pair_scope.eq('all')]
    vmax = max(int(part.n_pairs.max()), 1)
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.7), layout='constrained')
    for ax, group in zip(axes, GROUPS):
        sub = part.loc[part.salinity_group.eq(group)]
        grid = sub.pivot(index='time_bin', columns='space_bin', values='n_pairs').reindex(index=time_bins.lag_bin, columns=space_bins.lag_bin)
        img = ax.imshow(grid, origin='lower', aspect='auto', cmap='Blues', vmin=0, vmax=vmax)
        for _, row in sub.iterrows():
            ax.text(row.space_bin, row.time_bin, f'{int(row.n_pairs)} cặp\n{int(row.n_observations)} đv',
                    ha='center', va='center', fontsize=8, color='white' if row.n_pairs > vmax*.55 else '#111827')
        ax.set_title(f'Nhóm {GROUP_LABELS[group]}')
        set_st_axes(ax)
    fig.colorbar(img, ax=list(axes), shrink=.8, label='Số cặp (không phải số mẫu độc lập)')
    fig.suptitle(figure_title(model, kind, 'Độ phủ: số cặp và số đơn vị vị trí–ngày duy nhất'))
    save_st_figure(fig, model, kind, 'pair_counts', show)

def plot_st_slices(model=FOCUS_MODEL, kind=FOCUS_RESIDUAL, along='space',
                   metric='semivariance_standardized', show=True):
    if along not in {'space', 'time'}:
        raise ValueError('along phải là space hoặc time.')
    fixed_bins, varying_bins = (time_bins, space_bins) if along == 'space' else (space_bins, time_bins)
    fixed_key, varying_key = ('time_bin', 'space_bin') if along == 'space' else ('space_bin', 'time_bin')
    part = st_surfaces.loc[st_surfaces.model.eq(model) & st_surfaces.residual_kind.eq(kind) & st_surfaces.pair_scope.eq('all')]
    values = part.loc[metric_support(part, metric), metric]
    y_max = max(float(values.max()) * 1.08 if len(values) else 1., 1e-6)
    rows = (len(fixed_bins)+1)//2
    fig = plt.figure(figsize=(14, 5.2*rows), layout='constrained')
    outer = fig.add_gridspec(rows, 2)
    for index, (_, fixed) in enumerate(fixed_bins.iterrows()):
        inner = outer[index//2, index%2].subgridspec(2, 1, height_ratios=[4, 1.45])
        ax = fig.add_subplot(inner[0])
        ax_table = fig.add_subplot(inner[1])
        count_rows = []
        for group in GROUPS:
            sub = part.loc[part.salinity_group.eq(group) & part[fixed_key].eq(fixed.lag_bin)].set_index(varying_key).reindex(varying_bins.lag_bin)
            ax.plot(varying_bins.center, sub[metric], color=COLORS[group], marker=MARKERS[group],
                    linestyle=LINE_STYLES[group], label=GROUP_LABELS[group])
            count_rows.append([f'{int(n)}/{int(obs)}' for n, obs in zip(sub.n_pairs, sub.n_observations)])
        fixed_name, fixed_unit = ('u', 'ngày') if along == 'space' else ('h', 'km')
        ax.set_title(f'{fixed_name} thuộc {fixed.label} {fixed_unit}')
        ax.set_xlabel('Khoảng cách h (km)' if along == 'space' else 'Độ trễ u (ngày)')
        ax.set_ylabel('Tương quan phần dư' if metric == 'correlation' else 'Bán phương sai chuẩn hóa' if metric == 'semivariance_standardized' else 'Bán phương sai thô')
        if metric == 'correlation':
            ax.set_ylim(-1.05, 1.05)
            ax.axhline(0, color='#6b7280', lw=.8)
        else:
            ax.set_ylim(0, y_max)
        ax.grid(alpha=.2)
        ax.legend(fontsize=8)
        ax_table.axis('off')
        ax_table.set_title('Số cặp / số quan trắc duy nhất theo khoảng', fontsize=9, pad=4)
        table = ax_table.table(cellText=count_rows, rowLabels=[GROUP_LABELS[g] for g in GROUPS],
            colLabels=varying_bins.label.tolist(), cellLoc='center', loc='center', bbox=[.12, 0, .88, .88])
        table.auto_set_font_size(False)
        table.set_fontsize(8)
    fig.suptitle(figure_title(model, kind, f'Lát cắt {ST_LABELS[metric].lower()} theo {along}'))
    fig.supxlabel('Mọi giá trị hữu hạn đều được vẽ; số lượng tại từng điểm nằm trong bảng tương ứng bên dưới.', fontsize=9)
    save_st_figure(fig, model, kind, f'slices_{along}_{metric}', show)

def plot_st_contrasts(model=FOCUS_MODEL, kind=FOCUS_RESIDUAL, show=True):
    part = st_contrasts.loc[st_contrasts.model.eq(model) & st_contrasts.residual_kind.eq(kind) &
                           st_contrasts.pair_scope.eq('all') & st_contrasts.metric.eq('semivariance_standardized')]
    values = part.estimate.dropna().abs()
    maximum = max(float(values.max()) if len(values) else 1., 1e-6)
    norm = TwoSlopeNorm(vmin=-maximum, vcenter=0, vmax=maximum)
    cmap = plt.get_cmap('RdBu_r').copy()
    cmap.set_bad('#e5e7eb')
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.7), layout='constrained')
    for ax, (g1, g2) in zip(axes, PAIR_COMPARISONS):
        sub = part.loc[part.group_1.eq(g1) & part.group_2.eq(g2)]
        grid = sub.pivot(index='time_bin', columns='space_bin', values='estimate').reindex(index=time_bins.lag_bin, columns=space_bins.lag_bin)
        img = ax.imshow(np.ma.masked_invalid(grid.to_numpy(float)), origin='lower', aspect='auto', cmap=cmap, norm=norm)
        for _, row in sub.iterrows():
            label = f'{row.estimate:+.2f}' if np.isfinite(row.estimate) else 'KXĐ'
            if row.loo_stable_direction == True:
                label += '†'
            label += f'\nn={int(row.n_pairs_1)}/{int(row.n_pairs_2)}\nN={int(row.n_observations_1)}/{int(row.n_observations_2)}'
            color = 'white' if np.isfinite(row.estimate) and abs(row.estimate) > maximum*.6 else '#111827'
            ax.text(row.space_bin, row.time_bin, label, ha='center', va='center', fontsize=8, color=color)
        ax.set_title(f'{GROUP_LABELS[g2]} − {GROUP_LABELS[g1]}')
        set_st_axes(ax)
    fig.colorbar(img, ax=list(axes), shrink=.8, label='Chênh lệch bán phương sai chuẩn hóa')
    fig.suptitle(figure_title(model, kind, 'So sánh cùng ô (h, u) — mô tả, không phải kiểm định'))
    fig.supxlabel('n/N: số cặp/quan trắc; hai số mỗi dòng theo thứ tự nhóm 1/nhóm 2. †: LOO cùng dấu ≥80%, không phải kiểm định.', fontsize=9)
    save_st_figure(fig, model, kind, 'normalized_group_differences', show)

def plot_group_distribution(model=FOCUS_MODEL, kind=FOCUS_RESIDUAL, show=True):
    part = units.loc[units.model.eq(model)]
    samples = [part.loc[part.salinity_group.eq(g), KINDS[kind]].to_numpy() for g in GROUPS]
    fig, ax = plt.subplots(figsize=(8, 4.7), layout='constrained')
    label_kw = 'tick_labels' if 'tick_labels' in inspect.signature(ax.boxplot).parameters else 'labels'
    boxes = ax.boxplot(samples, **{label_kw: [GROUP_LABELS[g] for g in GROUPS]}, patch_artist=True)
    for box, g in zip(boxes['boxes'], GROUPS):
        box.set_facecolor(COLORS[g]); box.set_alpha(.65)
    ax.axhline(0, color='#6b7280', lw=.8)
    ax.set_ylabel('Phần dư (đơn vị độ mặn theo CSV)')
    ax.set_xlabel('Nhóm độ mặn')
    ax.set_title(figure_title(model, kind, 'Câu hỏi 1: độ lớn và phân bố phần dư'))
    save_st_figure(fig, model, kind, 'residual_distribution', show)

def plot_group_date_coverage(model=FOCUS_MODEL, kind=FOCUS_RESIDUAL, show=True):
    part = coverage_by_date.loc[coverage_by_date.model.eq(model)]
    pivot = part.pivot(index='salinity_group', columns='date', values='n_units').reindex(GROUPS).fillna(0)
    fig, ax = plt.subplots(figsize=(13, 3.5), layout='constrained')
    img = ax.imshow(pivot.to_numpy(), aspect='auto', cmap='Blues', vmin=0)
    ax.set_yticks(range(3), [GROUP_LABELS[g] for g in GROUPS])
    labels = [pd.Timestamp(d).strftime('%d/%m/%Y') for d in pivot.columns]
    ax.set_xticks(range(len(labels)), labels, rotation=60, ha='right', fontsize=8)
    ax.set_xlabel('Ngày quan trắc OOF')
    ax.set_ylabel('Nhóm độ mặn')
    fig.colorbar(img, ax=ax, label='Số đơn vị vị trí–ngày')
    ax.set_title(figure_title(model, kind, 'Độ phủ nhóm theo ngày: kiểm tra sự lệch giai đoạn/mùa'))
    save_st_figure(fig, model, kind, 'group_date_coverage', show)


### 14.1. Độ phủ cặp trong từng ô không gian–thời gian


In [ ]:
plot_st_counts()


### 14.2. Variogram thô — còn chứa ảnh hưởng độ lớn sai số


In [ ]:
plot_st_heatmaps(metric='semivariance')


### 14.3. Variogram chuẩn hóa — trọng tâm so sánh hình dạng


In [ ]:
plot_st_heatmaps()


### 14.4. Tương quan phần dư trong từng ô


In [ ]:
plot_st_heatmaps(metric='correlation')


### 14.5. Theo không gian, giữ từng khoảng thời gian


In [ ]:
plot_st_slices(along='space')


### 14.6. Theo thời gian, giữ từng khoảng cách không gian


In [ ]:
plot_st_slices(along='time')


### 14.7. Tương quan theo không gian ở từng khoảng thời gian


In [ ]:
plot_st_slices(along='space', metric='correlation')


### 14.8. Tương quan theo thời gian ở từng khoảng cách không gian


In [ ]:
plot_st_slices(along='time', metric='correlation')


### 14.9. Chênh lệch variogram chuẩn hóa giữa các nhóm


In [ ]:
plot_st_contrasts()


### 14.10. Độ nhạy khi chỉ xét cặp cùng outer fold


In [ ]:
plot_st_heatmaps(pair_scope='same_outer_fold')


### 14.11. Độ phủ nhóm theo ngày


In [ ]:
plot_group_date_coverage()


### 14.12. Câu hỏi 1 — Phân bố và độ lớn phần dư


In [ ]:
plot_group_distribution()


**Cách đọc:** xem số cặp/đơn vị và độ phủ theo ngày trước. Variogram chuẩn hóa thấp nghĩa là hai
đầu cặp ít khác nhau so với độ phân tán toàn nhóm. So sánh tốc độ biến đổi theo $h$ khi giữ $u$,
và theo $u$ khi giữ $h$, đối chiếu tương quan cặp và kết quả bỏ fold/cùng fold.
Khác biệt chỉ ở variogram thô có thể phản ánh biên độ sai số. Khác biệt sau chuẩn hóa là tín hiệu
đáng khảo sát, nhưng còn cần loại trừ ảnh hưởng mùa/vị trí/bin rộng và thay đổi mô hình OOF.
Không tự suy ra range, separability hay một họ kernel mới từ vài ô/đường cong.
Trên heatmap chênh lệch, † là cùng dấu ở ít nhất 80% lần bỏ fold tính được chênh lệch hữu hạn và đủ số lần theo cấu hình,
không phải ký hiệu có ý nghĩa thống kê. Các điểm lát cắt được đặt ở trung điểm bin dùng chung.
Bảng dưới từng lát cắt ghi số cặp / số quan trắc duy nhất. Một quan trắc có thể góp mặt ở nhiều ô; không cộng N giữa các ô.


## 15. Kiểm tra tính đúng và xuất hình cho các mô hình khác
Các kiểm tra bao gồm chronology OOF, công thức phần dư, nhóm, tính duy nhất của cặp,
bin 0, cặp khác cả vị trí lẫn ngày, và công thức variogram chuẩn hóa.


In [ ]:
assert not oof_rows.duplicated(['model', 'sample_id']).any()
for meta in fold_log:
    tr, va = np.asarray(meta['train_indices']), np.asarray(meta['validation_indices'])
    assert not set(tr) & set(va)
    assert dates[tr].max() < dates[va].min()
    assert pd.Timestamp(meta['fit_end']) < pd.Timestamp(meta['prediction_start'])
for model in MODEL_NAMES:
    part = oof_rows.loc[oof_rows.model.eq(model)]
    mask = part.valid_oof
    assert mask.sum() == sum(len(va) for _, va in outer_splits)
    for kind, col in KINDS.items():
        pred = 'oof_tree' if kind == 'tree' else 'oof_blend'
        np.testing.assert_allclose(part.loc[mask, col], part.loc[mask, 'y_true']-part.loc[mask, pred])
    cut = threshold_map[model]
    actual_groups = valid_rows.loc[valid_rows.model.eq(model), 'salinity_group'].to_numpy()
    np.testing.assert_array_equal(actual_groups, assign_group(part.loc[mask, GROUP_BY], cut['q1'], cut['q2']))
assert (geometry.i < geometry.j).all()
assert not geometry.duplicated(['i', 'j']).any()
assert not st_pairs.duplicated(['model', 'i', 'j']).any()
assert geometry.loc[geometry.space_bin.eq(0), 'distance_km'].eq(0).all()
assert geometry.loc[geometry.time_bin.eq(0), 'time_lag_days'].eq(0).all()
assert not ((geometry.space_bin == 0) & (geometry.time_bin == 0)).any()
for model in MODEL_NAMES:
    p = st_pairs.loc[st_pairs.model.eq(model)]
    point = units.loc[units.model.eq(model)].sort_values('unit_id')
    labels = point.salinity_group.to_numpy()
    np.testing.assert_array_equal(labels[p.i], labels[p.j])
assert st_surfaces.correlation.dropna().between(-1, 1).all()
valid = st_surfaces.semivariance_standardized.notna()
np.testing.assert_allclose(st_surfaces.loc[valid, 'semivariance_standardized'],
    st_surfaces.loc[valid, 'semivariance'] / st_surfaces.loc[valid, 'group_residual_sd']**2)
origin = st_surfaces.space_bin.eq(0) & st_surfaces.time_bin.eq(0)
assert st_surfaces.loc[origin, 'semivariance'].isna().all()
assert (st_surfaces.loc[origin, 'n_pairs'] == 0).all()
assert not st_surfaces.loc[~st_surfaces.coverage_supported, 'supported'].any()
assert (st_surfaces.loc[st_surfaces.pair_scope.eq('same_outer_fold'), 'fraction_cross_fold'].dropna() == 0).all()
print('Đạt: chronology OOF, phần dư, nhóm, cặp duy nhất, bin 0 và công thức variogram chuẩn hóa.')
if not geometry.pair_domain.eq('joint').any():
    print('LƯU Ý: dữ liệu không có cặp khác cả vị trí lẫn ngày; chưa khảo sát được cấu trúc hai chiều.')

if SAVE_ALL_MODEL_FIGURES:
    for model, kind in itertools.product(MODEL_NAMES, KINDS):
        if model == FOCUS_MODEL and kind == FOCUS_RESIDUAL:
            continue
        plot_st_counts(model, kind, show=False)
        for metric in ST_METRICS:
            plot_st_heatmaps(model, kind, metric=metric, show=False)
        for along in ['space', 'time']:
            for metric in ['semivariance_standardized', 'correlation']:
                plot_st_slices(model, kind, along=along, metric=metric, show=False)
        plot_st_contrasts(model, kind, show=False)
        plot_st_heatmaps(model, kind, pair_scope='same_outer_fold', show=False)
        plot_group_date_coverage(model, kind, show=False)
        plot_group_distribution(model, kind, show=False)


## 16. Kết quả đọc nhanh và hướng đánh giá kernel
Readout tách **độ lớn sai số**, **độ phủ so sánh cấu trúc**, và **độ nhạy**.
Không tự tuyên bố nhóm giống nhau khi thiếu dữ liệu, không suy ra cần kernel mới chỉ từ SD khác nhau.
Nếu cấu trúc khác có độ phủ và ổn định, bước tiếp theo là so sánh kernel hiện tại, kernel thay đổi
biên độ/noise theo độ mặn và kernel có thang tương quan phụ thuộc độ mặn bằng CV không gian–thời gian.
Khác thang tương quan chưa có nghĩa cần sáng tạo họ kernel mới; kernel mới còn phải hợp lệ PSD.


In [ ]:
diagnostic_config = {name: globals()[name] for name in ['DIAGNOSTIC_VERSION', 'GROUP_BY', 'GROUP_THRESHOLDS',
    'N_BOOTSTRAP', 'TIME_BLOCK_DAYS', 'SPACE_BLOCK_KM', 'AUTO_LAG_BINS', 'TIME_BIN_EDGES', 'SPACE_BIN_EDGES',
    'MIN_GROUP_N', 'MIN_PAIRS', 'MIN_DISTINCT_OBS', 'MIN_PAIR_STATIONS', 'MIN_PAIR_DATES', 'MIN_CLUSTERS',
    'MIN_BOOT_VALID_FRACTION', 'MIN_LOO_REPLICATES', 'NORMALIZED_GAMMA_EFFECT', 'MAX_GEOMETRY_PAIRS']}
save_json(RUN_DIR / 'diagnostic_config.json', diagnostic_config)
readout_rows = []
report_lines = [f'# Phần dư OOF: độ lớn và cấu trúc không gian–thời gian — {RUN_PROFILE}', '',
    f'- Chẩn đoán: {DIAGNOSTIC_VERSION}; chữ ký huấn luyện: {signature}.',
    f'- Phân nhóm theo {GROUP_BY}; ngưỡng riêng từng mô hình nếu không đặt GROUP_THRESHOLDS.',
    f'- Train: {len(train)} dòng, {train.date.nunique()} ngày; OOF mỗi mô hình: {int((oof_fold_id >= 1).sum())} dòng.',
    f'- Phân tích cặp: {len(reference)} đơn vị vị trí–ngày mỗi mô hình.',
    '- Các bảng cấu trúc là mô tả, không phải kiểm định; số cặp không phải số mẫu độc lập.',
    '- Heatmap/lát cắt vẽ mọi giá trị hữu hạn; n là số cặp, N là số quan trắc duy nhất.',
    '- Cờ supported/coverage_supported là tiêu chí đếm tùy chọn, không quyết định việc hiển thị hoặc tính chênh lệch.',
    '- LOO min–max không phải CI; ngưỡng/biên bin giữ nguyên, SD được tính lại sau bỏ fold.',
    '- So sánh cùng bin chưa bảo đảm cùng phân bố khoảng cách, vị trí, mùa hoặc mô hình OOF.', '']
if RUN_PROFILE == 'smoke':
    report_lines += ['**CHỈ CHẠY THỬ CODE, KHÔNG DÙNG LÀM KẾT QUẢ NGHIÊN CỨU.**', '']
if GROUP_BY == 'y_true':
    report_lines += ['**Nhóm theo thực đo chứa biến đích: khác biệt có thể do điều kiện hóa khi chọn nhóm.**', '']
for model, kind in itertools.product(MODEL_NAMES, KINDS):
    sub = st_contrasts.loc[st_contrasts.model.eq(model) & st_contrasts.residual_kind.eq(kind) &
        st_contrasts.pair_scope.eq('all') & st_contrasts.group_1.eq('low') & st_contrasts.group_2.eq('high') &
        st_contrasts.metric.eq('semivariance_standardized') & st_contrasts.pair_domain.eq('joint')]
    matched = sub.loc[sub.available]
    candidates = matched.loc[matched.estimate.abs().ge(NORMALIZED_GAMMA_EFFECT) & matched.loo_stable_direction.eq(True)]
    same = st_contrasts.loc[st_contrasts.model.eq(model) & st_contrasts.residual_kind.eq(kind) &
        st_contrasts.pair_scope.eq('same_outer_fold') & st_contrasts.group_1.eq('low') & st_contrasts.group_2.eq('high') &
        st_contrasts.metric.eq('semivariance_standardized') & st_contrasts.pair_domain.eq('joint') & st_contrasts.available]
    same_checked = candidates.merge(same[['space_bin', 'time_bin', 'estimate']], on=['space_bin', 'time_bin'], suffixes=('', '_same'))
    agreement = int((np.sign(same_checked.estimate) == np.sign(same_checked.estimate_same)).sum())
    status = ('Chưa có ô hai chiều tính được variogram chuẩn hóa cho cả nhóm cao và thấp.' if not len(matched) else
        'Có ô chênh lệch mô tả sau chuẩn hóa; cần đối chiếu tương quan, mùa/vị trí và kiểm tra kernel bằng CV.' if len(candidates) else
        'Chưa có tín hiệu mô tả đạt tiêu chí độ lớn và độ ổn định đã đặt; không đồng nghĩa hai nhóm giống nhau.')
    readout_rows.append(dict(model=model, residual_kind=kind, matched_joint_cells=len(matched),
        descriptive_candidate_cells=len(candidates), same_fold_checked_cells=len(same_checked), same_fold_same_sign_cells=agreement,
        interpretation=status))
    magnitude = unit_summary.loc[unit_summary.model.eq(model) & unit_summary.residual_kind.eq(kind)]
    report_lines += [f'## {model} | phần dư {kind}',
        '- Độ lớn sai số (SD / RMSE): ' + '; '.join(f'{GROUP_LABELS[r.group]} = {r.sd:.3f} / {r.rmse:.3f}' for r in magnitude.itertuples()),
        f'- Cấu trúc hai chiều cao–thấp: {len(matched)}/{len(sub)} ô tính được cho cả hai nhóm (không lọc theo ngưỡng đếm).',
        f'- Ô có |chênh lệch gamma chuẩn hóa| ≥ {NORMALIZED_GAMMA_EFFECT:g} và LOO cùng dấu ≥80% (ít nhất {MIN_LOO_REPLICATES} lần hợp lệ): {len(candidates)}.',
        f'- Trong số đó, kiểm tra được bằng cặp cùng fold: {len(same_checked)} ô; cùng dấu: {agreement} ô.',
        '- ' + status, '']
readout = pd.DataFrame(readout_rows)
save_csv(readout, 'st_readout_summary.csv')
display(readout.loc[readout.model.eq(FOCUS_MODEL)])
report_lines += ['## Diễn giải cho phát triển kernel',
    '- SD khác nhau mới gợi ý khác biên độ/noise; cần xem variogram chuẩn hóa và tương quan theo (h,u) để khảo sát cấu trúc.',
    '- Đọc độ phủ, centroid khoảng cách trong từng ô và kiểm tra biên bin/kích thước khối trước khi kết luận.',
    '- Không suy ra một họ kernel mới là cần thiết từ biểu đồ. So sánh các kernel bằng CV phù hợp và kiểm tra tính PSD.',
    '', '## File cần đọc',
    '- tables/st_variogram_surfaces.csv: mỗi nhóm × ô (h,u), cả all và same_outer_fold.',
    '- tables/st_group_contrasts.csv: chênh lệch cùng ô, số cặp, centroid khoảng cách và độ nhạy LOO.',
    '- tables/st_leave_one_fold_out_surfaces.csv và st_leave_one_fold_out_contrasts.csv.',
    '- tables/st_lag_bins.csv, st_group_coverage.csv, st_group_coverage_by_date.csv.',
    '- figures/st/: heatmap, lát cắt và biểu đồ phụ.',
    '- group_summary_station_date.csv và group_contrasts.csv: câu hỏi riêng về độ lớn sai số.']
(RUN_DIR / 'readout.md').write_text('\n'.join(report_lines), encoding='utf-8')
manifest = {'pipeline': PIPELINE_VERSION, 'diagnostic_version': DIAGNOSTIC_VERSION, 'profile': RUN_PROFILE,
    'signature': signature, 'diagnostic_signature': diagnostic_signature, 'models': MODEL_NAMES, 'checks': 'passed', 'group_by': GROUP_BY,
    'n_station_date_each_model': len(reference), 'n_unique_geometry_pairs': len(geometry),
    'n_joint_geometry_pairs': int(geometry.pair_domain.eq('joint').sum()),
    'figure_directory': str(INDIVIDUAL_FIGURE_DIR.relative_to(RUN_DIR)),
    'n_current_figures': len(list(INDIVIDUAL_FIGURE_DIR.glob('*.png'))),
    'training_config': training_config, 'diagnostic_config': diagnostic_config}
save_json(RUN_DIR / 'manifest.json', manifest)
archive = shutil.make_archive(str(OUTPUT_ROOT / 'oof_residual_diagnostics'), 'zip', root_dir=RUN_DIR)
import zipfile
with zipfile.ZipFile(archive, 'a', compression=zipfile.ZIP_DEFLATED) as bundle:
    for checkpoint in sorted(CACHE_DIR.iterdir()):
        if checkpoint.is_file() and checkpoint.suffix in {'.json', '.npz'}:
            bundle.write(checkpoint, 'checkpoints/' + checkpoint.name)
print('Kết quả:', RUN_DIR)
print('Hình:', INDIVIDUAL_FIGURE_DIR)
print('ZIP tải từ Kaggle Output:', archive)
display(Markdown('**Đọc trước:** heatmap/lát cắt mục 14, `readout.md`, `st_variogram_surfaces.csv` và '
                 '`st_group_contrasts.csv`. Thiếu độ phủ có nghĩa là chưa đủ dữ liệu, không phải hai nhóm giống nhau.'))
